# RAC vs MAC — A100 Proof-of-Performance
**Pinnacle Quantum Group — March 2026**

Rotate-Accumulate (RAC) vs Multiply-Accumulate (MAC/cuBLAS) benchmark on NVIDIA A100.

**Requirements:** Colab with A100 GPU runtime.

Go to **Runtime → Change runtime type → A100 GPU** before running.

In [ ]:
# Verify GPU
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader
import torch
assert torch.cuda.is_available(), "No GPU found — change runtime to A100"
props = torch.cuda.get_device_properties(0)
print(f"Device: {props.name}")
print(f"Compute: {props.major}.{props.minor}")
print(f"Memory: {props.total_memory // (1024**2)} MB")
print(f"SMs: {props.multi_processor_count}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# Clone RAC and build the CUDA extension (target only detected GPU — fast build)
!git clone https://github.com/Pinnacle-Quantum-Group/RAC.git /content/RAC 2>/dev/null || (cd /content/RAC && git pull)

import torch, os
# Only compile for the GPU we have — cuts build time from ~5min to ~30sec
cc = torch.cuda.get_device_properties(0)
arch = f"{cc.major}.{cc.minor}"
os.environ['TORCH_CUDA_ARCH_LIST'] = arch
print(f"Building for compute capability {arch} only...")

%cd /content/RAC/rac_torch
!pip install -e . 2>&1 | tail -5

# Make the extension importable from anywhere
import glob, sys
so_files = glob.glob('/content/RAC/rac_torch/rac_cuda_ext*.so') + \
           glob.glob('/content/RAC/rac_torch/build/lib*/rac_cuda_ext*.so')
if so_files:
    ext_dir = os.path.dirname(so_files[0])
    if ext_dir not in sys.path:
        sys.path.insert(0, ext_dir)
    print(f"Extension found: {so_files[0]}")
else:
    # Fallback: build inplace
    print("Rebuilding inplace...")
    !python setup.py build_ext --inplace 2>&1 | tail -3
    sys.path.insert(0, '/content/RAC/rac_torch')

%cd /content/RAC

In [ ]:
# Verify extension loads and passes correctness tests
import rac_cuda_ext
print("Extension loaded OK")

device = 'cuda'

# Test 1: Identity
A = torch.eye(4, device=device, dtype=torch.float32)
B = torch.tensor([[1.],[2.],[3.],[4.]], device=device, dtype=torch.float32)
C = rac_cuda_ext.matmul_forward(A, B)
torch.cuda.synchronize()
assert (C - B).abs().max().item() < 1e-5, "Identity test FAILED"
print("Identity:       PASS")

# Test 2: 2x2
A = torch.tensor([[1., 2.], [3., 4.]], device=device)
B = torch.tensor([[5., 6.], [7., 8.]], device=device)
C = rac_cuda_ext.matmul_forward(A, B)
C_ref = torch.matmul(A, B)
torch.cuda.synchronize()
assert (C - C_ref).abs().max().item() < 1e-5, "2x2 test FAILED"
print("2x2 matmul:     PASS")

# Test 3: linear_forward (NT kernel path)
W = torch.randn(64, 128, device=device)
x = torch.randn(8, 128, device=device)
bias = torch.zeros(64, device=device)
y_rac = rac_cuda_ext.linear_forward(x, W, bias)
y_ref = torch.nn.functional.linear(x, W, bias)
torch.cuda.synchronize()
err = (y_rac - y_ref).abs().max().item()
assert err < 0.01, f"linear_forward FAILED: err={err}"
print(f"linear_forward: PASS (err={err:.2e})")

# Test 4: Large matmul
for sz in [256, 512, 1024, 2048, 4096]:
    A = torch.randn(sz, sz, device=device)
    B = torch.randn(sz, sz, device=device)
    C = rac_cuda_ext.matmul_forward(A, B)
    C_ref = torch.matmul(A, B)
    torch.cuda.synchronize()
    err = (C - C_ref).abs().max().item()
    status = "PASS" if err < 0.1 else "FAIL"
    print(f"{sz}x{sz} matmul: {status} (err={err:.2e})")

# Test 5: linear_forward at scale
for out_f, in_f, batch in [(512, 512, 32), (1024, 1024, 32), (2048, 2048, 8), (4096, 4096, 8)]:
    W = torch.randn(out_f, in_f, device=device)
    x = torch.randn(batch, in_f, device=device)
    bias = torch.zeros(out_f, device=device)
    y_rac = rac_cuda_ext.linear_forward(x, W, bias)
    y_ref = torch.nn.functional.linear(x, W, bias)
    torch.cuda.synchronize()
    err = (y_rac - y_ref).abs().max().item()
    status = "PASS" if err < 0.01 else "FAIL"
    print(f"linear {batch}x{in_f} -> {out_f}: {status} (err={err:.2e})")

print("\nAll correctness tests passed.")

In [ ]:
# ── Benchmark: RAC vs MAC (cuBLAS) on A100 ──

import torch
import torch.nn as nn
import torch.nn.functional as F
import rac_cuda_ext
import math

device = torch.device('cuda')
props = torch.cuda.get_device_properties(0)


class RACLinear(nn.Module):
    """RAC linear layer using compiled NT kernel."""
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None

    def forward(self, x):
        bias_t = self.bias if self.bias is not None else torch.tensor([], device=x.device)
        return rac_cuda_ext.linear_forward(x, self.weight, bias_t)


class RACLinearFused(nn.Module):
    """RAC linear + GELU fused in one kernel pass."""
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None
        self._has_fused = hasattr(rac_cuda_ext, 'fused_linear_forward')

    def forward(self, x):
        if self._has_fused:
            bias_t = self.bias if self.bias is not None else torch.tensor([], device=x.device)
            return rac_cuda_ext.fused_linear_forward(x, self.weight, bias_t, 2)  # 2=GELU
        out = F.linear(x, self.weight, self.bias)
        return F.gelu(out)


def benchmark_layer(layer, x, n_warmup=20, n_iters=200):
    """Returns (latency_ms, output)."""
    with torch.no_grad():
        for _ in range(n_warmup):
            _ = layer(x)
    torch.cuda.synchronize()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    with torch.no_grad():
        for _ in range(n_iters):
            out = layer(x)
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / n_iters, out


# ── Configuration ──
sizes = [256, 512, 1024, 2048, 4096]
batches = [1, 8, 32]

print("=" * 100)
print("  RAC vs MAC — A100 Proof-of-Performance")
print("  Pinnacle Quantum Group — March 2026")
print("=" * 100)
print(f"  Device: {props.name}  |  SMs: {props.multi_processor_count}  |  Memory: {props.total_memory // (1024**2)} MB")
print(f"  PyTorch: {torch.__version__}")
print("=" * 100)
print()
print(f"{'Size':>6} {'Batch':>5} {'MAC (ms)':>10} {'RAC (ms)':>10} {'RAC+GELU':>10}"
      f" {'RAC/MAC':>8} {'FUSED/MAC':>10} {'MaxErr':>10}")
print("─" * 100)

results = []

for size in sizes:
    for batch in batches:
        # Build layers with identical weights
        mac = nn.Linear(size, size, bias=True).to(device).eval()
        x = torch.randn(batch, size, device=device)

        rac = RACLinear(size, size, bias=True).to(device).eval()
        with torch.no_grad():
            rac.weight.copy_(mac.weight)
            rac.bias.copy_(mac.bias)

        fused = RACLinearFused(size, size, bias=True).to(device).eval()
        with torch.no_grad():
            fused.weight.copy_(mac.weight)
            fused.bias.copy_(mac.bias)

        lat_mac, out_mac = benchmark_layer(mac, x)
        lat_rac, out_rac = benchmark_layer(rac, x)
        lat_fused, out_fused = benchmark_layer(fused, x)

        max_err = (out_mac.float() - out_rac.float()).abs().max().item()
        speedup_rac = lat_mac / lat_rac if lat_rac > 0 else 0
        speedup_fused = lat_mac / lat_fused if lat_fused > 0 else 0

        print(f"{size:6d} {batch:5d} {lat_mac:10.3f} {lat_rac:10.3f} {lat_fused:10.3f}"
              f" {speedup_rac:7.2f}x {speedup_fused:9.2f}x {max_err:10.2e}")

        results.append({
            'size': size, 'batch': batch,
            'lat_mac': lat_mac, 'lat_rac': lat_rac, 'lat_fused': lat_fused,
            'speedup_rac': speedup_rac, 'speedup_fused': speedup_fused,
            'max_err': max_err,
        })

print("─" * 100)

In [ ]:
# ── Summary ──

print("\n" + "=" * 100)
print("  SUMMARY")
print("=" * 100)

rac_wins = [r for r in results if r['speedup_rac'] > 1.0]
fused_wins = [r for r in results if r['speedup_fused'] > 1.0]
# Best of RAC or FUSED per config
either_wins = [r for r in results if r['speedup_rac'] > 1.0 or r['speedup_fused'] > 1.0]

print(f"\n  RAC vs MAC (Rotate-Accumulate vs Multiply-Accumulate):")
print(f"    RAC wins at {len(rac_wins)}/{len(results)} configurations")
if rac_wins:
    best = max(rac_wins, key=lambda r: r['speedup_rac'])
    print(f"    Best: {best['speedup_rac']:.2f}x at {best['size']}x{best['size']} batch={best['batch']}")

print(f"\n  RAC+GELU vs MAC:")
print(f"    RAC+GELU wins at {len(fused_wins)}/{len(results)} configurations")
if fused_wins:
    best_f = max(fused_wins, key=lambda r: r['speedup_fused'])
    print(f"    Best: {best_f['speedup_fused']:.2f}x at {best_f['size']}x{best_f['size']} batch={best_f['batch']}")

print(f"\n  Combined (best of RAC or RAC+GELU per config):")
print(f"    Wins at {len(either_wins)}/{len(results)} configurations")

# Accuracy
max_errs = [r['max_err'] for r in results]
print(f"\n  Numerical accuracy:")
print(f"    Max error across all configs: {max(max_errs):.2e}")
print(f"    Within float32 tolerance: {'YES' if max(max_errs) < 0.1 else 'NO'}")

print("\n" + "=" * 100)

In [ ]:
# ── Plots ──

import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'RAC vs MAC — {props.name}', fontsize=14, fontweight='bold')

for bi, batch in enumerate(batches):
    ax = axes[bi]
    subset = [r for r in results if r['batch'] == batch]
    szs = [r['size'] for r in subset]
    x_pos = np.arange(len(szs))
    w = 0.3

    sp_rac = [r['speedup_rac'] for r in subset]
    sp_fused = [r['speedup_fused'] for r in subset]

    colors_r = ['#2ecc71' if s >= 1.0 else '#e74c3c' for s in sp_rac]
    colors_f = ['#27ae60' if s >= 1.0 else '#e67e22' for s in sp_fused]

    ax.bar(x_pos - w/2, sp_rac, w, color=colors_r, alpha=0.8, label='RAC')
    ax.bar(x_pos + w/2, sp_fused, w, color=colors_f, alpha=0.8, label='RAC+GELU')
    ax.axhline(y=1.0, color='black', linestyle='--', linewidth=1)
    ax.set_xticks(x_pos)
    ax.set_xticklabels([str(s) for s in szs])
    ax.set_xlabel('Matrix Size (NxN)')
    ax.set_ylabel('Speedup vs cuBLAS')
    ax.set_title(f'Batch={batch}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('rac_a100_benchmark.png', dpi=150)
plt.show()
print("Plot saved: rac_a100_benchmark.png")

In [ ]:
# ── Detailed per-config table ──

print(f"\n{'Size':>6} {'Batch':>5} │ {'MAC':>8} {'RAC':>8} {'FUSED':>8} │"
      f" {'RAC/MAC':>8} {'FUSED/MAC':>10} {'Best':>8} │ {'MaxErr':>10}")
print("─" * 95)

for r in results:
    best_speedup = max(r['speedup_rac'], r['speedup_fused'])
    best_label = 'RAC' if r['speedup_rac'] >= r['speedup_fused'] else 'FUSED'
    winner = '✓' if best_speedup > 1.0 else ''
    print(f"{r['size']:6d} {r['batch']:5d} │"
          f" {r['lat_mac']:7.3f}ms {r['lat_rac']:7.3f}ms {r['lat_fused']:7.3f}ms │"
          f" {r['speedup_rac']:7.2f}x {r['speedup_fused']:9.2f}x"
          f" {best_speedup:6.2f}x {winner:>1} │ {r['max_err']:10.2e}")

print("─" * 95)
total_wins = sum(1 for r in results if max(r['speedup_rac'], r['speedup_fused']) > 1.0)
print(f"\nRAC or RAC+GELU beats cuBLAS at {total_wins}/{len(results)} configurations")